In [ ]:
import os
import sys

os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import tqdm.auto
import tqdm.std

tqdm.auto.tqdm = tqdm.std.tqdm
tqdm.auto.trange = tqdm.std.trange

import logging
from pathlib import Path

import pandas as pd

from drzavni_posao import (
    CharacterAssignerConfig,
    DiarizerConfig,
    EmbedderConfig,
    MediaStore,
    MlflowTracker,
    PipelineConfig,
    Stage,
    TranscriberConfig,
    run_pipeline,
)
from drzavni_posao.comparison import (
    cast_accuracy,
    compare_all,
    load_transcripts,
    scribe_transcripts,
    text_of,
)
from drzavni_posao.gallery import load_speaker_samples

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("mlflow.system_metrics").setLevel(logging.WARNING)

logger = logging.getLogger("drzavni_posao.pipeline")


In [ ]:
EXPERIMENT_NAME = (
    "/Users/luka.erceg@databricks.com/drzavni_posao/transcription"
)
AUDIO_PATHS = "/Volumes/luka_erceg/drzavni_posao/audios"
TRANSCRIPT_PATH = (
    "/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/transcripts"
)
GALLERY_PATH = Path("/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/labels/gallery_base.json")

HF_TOKEN = dbutils.secrets.get(scope="luka-erceg", key="hf_token")

SCRIBE_DIR = f"{TRANSCRIPT_PATH}/elevenlabs"

DEVICE = "cuda"
COMPUTE_TYPE = "bfloat16"
EMBEDDER_MODEL = "pyannote/wespeaker-voxceleb-resnet34-LM"

ASR_MODELS = {
    "sr-cmb": "/Volumes/workspace/default/luka_erceg_models/whisper-sr-ct2",
    "large-v3": "large-v3",
    "large-v3-turbo": "large-v3-turbo",
}
ALIGN_MODELS = {
    "classla": "classla/wav2vec2-xls-r-parlaspeech-hr",
}

CAST_HOTWORDS = (
    "Đorđe Čvarkov, Dragan Torbica, Boškić, Dobrosavljev, "
    "Marijanović, Živan Radosavljević, Frau Šilovička"
)

NGRAM5 = {"no_repeat_ngram_size": 5, "repetition_penalty": 1.0}
DECODING_VARIANTS = {
    "baseline": {},
    "ngram3": {"no_repeat_ngram_size": 3, "repetition_penalty": 1.0},
    "ngram5": NGRAM5,
    "ngram5-numerals": {**NGRAM5, "suppress_numerals": True},
    "ngram5-hotwords": {**NGRAM5, "hotwords": CAST_HOTWORDS},
    "ngram5-beam10": {**NGRAM5, "beam_size": 10},
}

ALIGN_MODEL_KEY = "classla"

RUNS = [
    ("large-v3", "ngram5"),
    ("large-v3", "baseline"),
    ("large-v3", "ngram3"),
    ("large-v3", "ngram5-numerals"),
    ("large-v3", "ngram5-hotwords"),
    ("large-v3", "ngram5-beam10"),
]

EPISODES = set(scribe_transcripts(SCRIBE_DIR))

tracker = MlflowTracker(EXPERIMENT_NAME)


In [ ]:
SPEAKER_SAMPLES = load_speaker_samples(GALLERY_PATH)


In [ ]:
media = MediaStore(
    AUDIO_PATHS,
    TRANSCRIPT_PATH,
    trim_start=25,
    trim_end=7,
)

diarizer_config = DiarizerConfig(
    device=DEVICE,
    hf_token=HF_TOKEN,
    min_speakers=3,
    max_speakers=3,
)
character_assigner_config = CharacterAssignerConfig(
    min_similarity=0.1,
    embedder=EmbedderConfig(
        device=DEVICE,
        hf_token=HF_TOKEN,
        embed_model=EMBEDDER_MODEL,
        min_segment_seconds=0.01,
    ),
    speaker_samples=SPEAKER_SAMPLES,
)


def transcription_config(model_key: str, decoding_key: str) -> TranscriberConfig:
    return TranscriberConfig(
        device=DEVICE,
        compute_type=COMPUTE_TYPE,
        model_name=ASR_MODELS[model_key],
        language="sr",
        align_model_name=ALIGN_MODELS[ALIGN_MODEL_KEY],
        asr_options=DECODING_VARIANTS[decoding_key],
    )


In [ ]:
BAKEOFF_CONFIGS = {
    variant: PipelineConfig.from_media(
        media,
        transcription=transcription_config(*variant),
    )
    for variant in RUNS
}

BAKEOFF_RUN_IDS = {
    variant: run_pipeline(
        tracker,
        media,
        config,
        EPISODES,
        through=Stage.TRANSCRIPTION,
        tags={"experiment": "model-bakeoff"},
    )
    for variant, config in BAKEOFF_CONFIGS.items()
}


In [ ]:
HOTWORDS_VARIANT = ("large-v3", "ngram5-hotwords")
HOTWORDS_CONFIG = BAKEOFF_CONFIGS[HOTWORDS_VARIANT]
ALL_EPISODES = {episode.key for episode in media.episodes()}

HOTWORDS_RUN_ID = run_pipeline(
    tracker,
    media,
    HOTWORDS_CONFIG,
    ALL_EPISODES,
    through=Stage.TRANSCRIPTION,
)


In [ ]:
HOTWORDS_ASSIGNER_CONFIG = PipelineConfig.from_media(
    media,
    transcription=HOTWORDS_CONFIG.transcription,
    character_assigner=character_assigner_config,
)

tracker.fork_run(
    HOTWORDS_RUN_ID,
    HOTWORDS_ASSIGNER_CONFIG.generated_name(),
    HOTWORDS_ASSIGNER_CONFIG.to_tracking_dict(),
    copy_until=Stage.TRANSCRIPTION,
)
HOTWORDS_ASSIGNER_RUN_ID = run_pipeline(
    tracker,
    media,
    HOTWORDS_ASSIGNER_CONFIG,
    ALL_EPISODES,
    through=Stage.CHARACTER_ASSIGNER,
    tags={"experiment": "character-assigner"},
)


In [ ]:
HOTWORDS_SPEAKERS_CONFIG = PipelineConfig.from_media(
    media,
    transcription=HOTWORDS_CONFIG.transcription,
    diarization=diarizer_config,
    character_assigner=character_assigner_config,
)

tracker.fork_run(
    HOTWORDS_RUN_ID,
    HOTWORDS_SPEAKERS_CONFIG.generated_name(),
    HOTWORDS_SPEAKERS_CONFIG.to_tracking_dict(),
    copy_until=Stage.TRANSCRIPTION,
)
HOTWORDS_SPEAKERS_RUN_ID = run_pipeline(
    tracker,
    media,
    HOTWORDS_SPEAKERS_CONFIG,
    ALL_EPISODES,
    through=Stage.CHARACTER_ASSIGNER,
    tags={"experiment": "speakers"},
)

In [ ]:
# Score every run against Scribe. Scribe is not ground truth, so these are
# coverage-relative-to-a-second-system numbers: `agreement` is the share of
# Scribe's words the run also reproduced, `word_deficit` how much shorter its
# transcript is, `in_vad_gaps` how many reference words land in stretches it
# emitted nothing for.
#
# The per-name columns are name RECALL — correctly-spelled mentions over the
# mentions Scribe found. A plain spelling ratio hides dropped names, since a
# name that is never transcribed cannot be misspelled, and for the search index
# a missing mention costs the same as a mangled one.
references = scribe_transcripts(SCRIBE_DIR)


def episode_comparisons(model_key: str, decoding_key: str):
    run_id = BAKEOFF_RUN_IDS[(model_key, decoding_key)]
    run_name = tracker.client.get_run(run_id).data.tags["mlflow.runName"]
    return compare_all(
        load_transcripts(f"{TRANSCRIPT_PATH}/{run_name}"),
        references,
    )


def score(model_key: str, decoding_key: str) -> dict:
    comparisons = episode_comparisons(model_key, decoding_key)
    metrics = [comparison.metrics() for comparison in comparisons]
    reference_words = sum(metric["reference_words"] for metric in metrics)
    words = sum(metric["hypothesis_words"] for metric in metrics)

    spoken = cast_accuracy(
        " ".join(text_of(c.hypothesis) for c in comparisons)
    )
    expected = cast_accuracy(
        " ".join(text_of(c.reference) for c in comparisons)
    )
    return {
        "model": model_key,
        "episodes": len(comparisons),
        "words": words,
        "word_deficit": 1 - words / reference_words,
        "agreement": sum(len(c.matched) for c in comparisons) / reference_words,
        "in_vad_gaps": sum(metric["words_in_vad_gaps"] for metric in metrics),
        **{
            name: (
                stats["mentions"] * stats["well_formed"] / expected[name]["mentions"]
                if expected[name]["mentions"]
                else float("nan")
            )
            for name, stats in spoken.items()
        },
    }


scores = pd.DataFrame(
    {decoding_key: score(model, decoding_key) for model, decoding_key in RUNS}
).T
scores


In [ ]:
# Drill-down behind the name-recall columns: how many mentions each run found
# against how many Scribe found, how many of those were spelled acceptably, and
# the actual misspellings. This is what separates "found fewer" from "spelled
# worse" when a recall number moves.
detail = []
for model_key, decoding_key in RUNS:
    comparisons = episode_comparisons(model_key, decoding_key)
    spoken = cast_accuracy(
        " ".join(text_of(c.hypothesis) for c in comparisons)
    )
    expected = cast_accuracy(
        " ".join(text_of(c.reference) for c in comparisons)
    )
    for name, stats in spoken.items():
        detail.append(
            {
                "run": decoding_key,
                "name": name,
                "found": stats["mentions"],
                "in_reference": expected[name]["mentions"],
                "well_formed": stats["well_formed"],
                "variants": ", ".join(stats["variants"]) or "—",
            }
        )

pd.DataFrame(detail).set_index(["run", "name"])


In [ ]:
SR_CMB_VARIANT = ("sr-cmb", "ngram5")
SR_CMB_CONFIG = PipelineConfig.from_media(
    media,
    transcription=transcription_config(*SR_CMB_VARIANT),
)
SR_CMB_RUN_ID = run_pipeline(
    tracker,
    media,
    SR_CMB_CONFIG,
    ALL_EPISODES,
    through=Stage.TRANSCRIPTION,
    tags={"experiment": "source-transcription"},
)


In [ ]:
SR_CMB_SPEAKERS_CONFIG = PipelineConfig.from_media(
    media,
    transcription=SR_CMB_CONFIG.transcription,
    diarization=diarizer_config,
    character_assigner=character_assigner_config,
)

tracker.fork_run(
    SR_CMB_RUN_ID,
    SR_CMB_SPEAKERS_CONFIG.generated_name(),
    SR_CMB_SPEAKERS_CONFIG.to_tracking_dict(),
    copy_until=Stage.TRANSCRIPTION,
)
SR_CMB_SPEAKERS_RUN_ID = run_pipeline(
    tracker,
    media,
    SR_CMB_SPEAKERS_CONFIG,
    ALL_EPISODES,
    through=Stage.CHARACTER_ASSIGNER,
    tags={"experiment": "speakers"},
)


In [ ]:
# Forked transcription metrics must remain identical to the source.
source_run_name = tracker.client.get_run(SR_CMB_RUN_ID).data.tags["mlflow.runName"]
speaker_run_name = tracker.client.get_run(
    SR_CMB_SPEAKERS_RUN_ID
).data.tags["mlflow.runName"]
source_counts = tracker.episode_metrics(
    source_run_name,
    "transcription.word_count",
)
speaker_counts = tracker.episode_metrics(
    speaker_run_name,
    "transcription.word_count",
)
compared = source_counts.keys() & speaker_counts.keys()

drift = pd.DataFrame(
    [
        {
            "episode": key,
            "source": source_counts[key],
            "speakers": speaker_counts[key],
        }
        for key in sorted(compared, key=int)
        if source_counts[key] != speaker_counts[key]
    ]
)
print(f"compared {len(compared)} episodes, {len(drift)} mismatched")
drift


In [ ]:
SR_CMB_ASSIGNER_CONFIG = PipelineConfig.from_media(
    media,
    transcription=SR_CMB_CONFIG.transcription,
    character_assigner=character_assigner_config,
)

tracker.fork_run(
    SR_CMB_RUN_ID,
    SR_CMB_ASSIGNER_CONFIG.generated_name(),
    SR_CMB_ASSIGNER_CONFIG.to_tracking_dict(),
    copy_until=Stage.TRANSCRIPTION,
)
SR_CMB_ASSIGNER_RUN_ID = run_pipeline(
    tracker,
    media,
    SR_CMB_ASSIGNER_CONFIG,
    ALL_EPISODES,
    through=Stage.CHARACTER_ASSIGNER,
    tags={"experiment": "character-assigner"},
)
